# Task 14: Custom CUDA-Accelerated Swish Activation Layer Integration

**Objective:** Build high-performance GPU kernels to accelerate non-linear activation layers, compiling PyTorch C++ bindings dynamically.

### Formula
$$\text{Swish}(x) = x \cdot \text{Sigmoid}(x)$$
$$\text{Swish}'(x) = \text{Sigmoid}(x) + x \cdot \text{Sigmoid}(x) \cdot (1 - \text{Sigmoid}(x))$$

This folder contains:
1. C++ bindings (`swish_cpp.cpp`)
2. CUDA source (`swish_kernel.cu`)
3. Compiler setup (`setup.py`)

The notebook contains custom PyTorch Autograd fallbacks and benchmarks.

In [ ]:
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt

# 1. Custom Autograd Function (Python Fallback version for numerical correctness and testing)
class CustomSwishFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * torch.sigmoid(x)
        
    @staticmethod
    def backward(ctx, dy):
        x, = ctx.saved_tensors
        sig = torch.sigmoid(x)
        dx = dy * (sig + x * sig * (1.0 - sig))
        return dx

class CustomSwish(nn.Module):
    def forward(self, x):
        return CustomSwishFunction.apply(x)

# 2. Benchmark code against PyTorch Native
x_eval = torch.randn(5000, 5000, device='cuda' if torch.cuda.is_available() else 'cpu')

# Evaluation
swish_custom = CustomSwish()
swish_native = nn.SiLU() # PyTorch's native Swish implementation

t0 = time.time()
for _ in range(100):
    y_custom = swish_custom(x_eval)
t1 = time.time()

t2 = time.time()
for _ in range(100):
    y_native = swish_native(x_eval)
t3 = time.time()

print(f"Custom Swish (100 passes): {t1 - t0:.6f} seconds")
print(f"Native SiLU  (100 passes): {t3 - t2:.6f} seconds")